# Bài 3.4 · Training loop trên ảnh thật

**Câu hỏi của buổi này:** một vòng huấn luyện đúng trông thế nào khi chạy thật, và làm sao biết nó đúng: loss bắt đầu ở đâu, khi nào dừng, giữ checkpoint nào, lưu ra sao để nạp lại cho cùng kết quả?

Dữ liệu là **Fashion-MNIST**: ảnh xám 28×28 của 10 loại quần áo, giày, túi. Để thấy rõ overfitting, ta chỉ cho model học **10 000 ảnh**; 10 000 ảnh khác làm validation, 10 000 ảnh test để riêng.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng ba tập, ảnh mẫu của 10 lớp |
| 2. Kiểm quy trình trên một batch | loss bắt đầu gần ln 10 rồi xuống gần 0 |
| 3. Vòng huấn luyện đầy đủ | bảng từng epoch, đường loss và accuracy train/val |
| 4. Chọn checkpoint | bảng test của checkpoint tốt nhất và epoch cuối |
| 5. Trung bình đúng theo mẫu | bảng: trung bình theo batch lệch so với theo mẫu |
| 6. Lưu và nạp lại | bảng nội dung checkpoint, kiểm dự đoán trùng khớp |
| 7. Tái lập | ba đường loss: hai seed giống nhau trùng khít |
| 8. Model nhìn ảnh test | ảnh đoán đúng/sai, những lỗi tự tin nhất |
| 9. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 2 phút. Bài giảng: [ai-canuc80k · bài 3.4](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/3.4).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, copy, random
from pathlib import Path
import torch, torchvision
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

seed_all(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
table(pd.DataFrame([('torch', torch.__version__), ('torchvision', torchvision.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Ảnh được đổi về số thực trong [0, 1] và duỗi thành vector 784 số. Ba tập không chung ảnh nào.

In [ ]:
VI = ['áo phông', 'quần dài', 'áo len', 'váy liền', 'áo khoác', 'xăng-đan', 'áo sơ mi', 'giày thể thao', 'túi xách', 'bốt']
with quiet():
    train_full = torchvision.datasets.FashionMNIST('/tmp/fmnist', train=True, download=True)
    test_set = torchvision.datasets.FashionMNIST('/tmp/fmnist', train=False, download=True)
X_all = train_full.data.float().div(255).view(-1, 784); y_all = train_full.targets
X_te = test_set.data.float().div(255).view(-1, 784); y_te = test_set.targets

perm = torch.randperm(60000, generator=torch.Generator().manual_seed(0))
tr_idx, va_idx = perm[:10000], perm[50000:]
X_tr, y_tr, X_va, y_va = X_all[tr_idx], y_all[tr_idx], X_all[va_idx], y_all[va_idx]

table(pd.DataFrame([('train', len(y_tr), 'model học'), ('validation', len(y_va), 'chọn epoch, chọn checkpoint'), ('test', len(y_te), 'chấm một lần ở cuối')],
                   columns=['tập', 'số ảnh', 'dùng để']), 'Ba tập')
check('train và validation không chung ảnh', len(set(tr_idx.tolist()) & set(va_idx.tolist())) == 0)
check(f'mỗi lớp trong train có {np.bincount(y_tr).min()}–{np.bincount(y_tr).max()} ảnh', np.bincount(y_tr).min() > 900)

fig, axes = plt.subplots(3, 10, figsize=(14, 4.6))
for k in range(10):
    ids = torch.nonzero(y_tr == k).flatten()[:3]
    for r in range(3):
        axes[r, k].imshow(X_tr[ids[r]].view(28, 28), cmap='gray_r'); axes[r, k].axis('off')
    axes[0, k].set_title(VI[k], fontsize=10)
fig.suptitle('Mười lớp, mỗi lớp ba ảnh train', y=1.0)
plt.tight_layout(); plt.show()

## 2. Kiểm toàn bộ quy trình trên một batch

Trước khi huấn luyện thật, cho model học đi học lại **một batch 64 ảnh**. Nếu code đúng, hai điều phải xảy ra:

- loss lúc đầu gần $\ln 10 \approx 2.303$, vì model chưa học gì thì đoán gần đều 10 lớp;
- loss xuống gần 0, vì 64 ảnh thì model đủ sức học thuộc.

Loss không xuống là có lỗi ở nhãn, ở loss hay ở bước cập nhật, chưa cần chạy cả tập để biết.

In [ ]:
def make_model():
    return nn.Sequential(nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10)).to(device)

seed_all(0)
model = make_model(); opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
xb, yb = X_tr[:64].to(device), y_tr[:64].to(device)
one_batch = []
for step in range(150):
    loss = F.cross_entropy(model(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
    one_batch.append(loss.item())

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(one_batch, color=C['blue']); ax.set_yscale('log')
ax.axhline(np.log(10), color=C['gray'], ls='--'); ax.text(60, np.log(10) * 1.35, 'ln 10 = 2.303', color=C['gray'], fontsize=9)
ax.set(xlabel='bước cập nhật', ylabel='loss (thang log)', title='Học thuộc một batch 64 ảnh')
plt.tight_layout(); plt.show()
check(f'loss bước đầu {one_batch[0]:.3f}, gần ln 10', abs(one_batch[0] - np.log(10)) < 0.3)
check(f'loss sau 150 bước {one_batch[-1]:.4f}, gần 0', one_batch[-1] < 0.01)

## 3. Vòng huấn luyện đầy đủ

Mỗi epoch có hai pha:

- **train**: `model.train()`, đi qua mọi batch theo thứ tự xáo, tính loss, `zero_grad` → `backward` → `step`;
- **validation**: `model.eval()`, tắt gradient, chỉ đo.

Loss và accuracy cộng dồn **theo từng ảnh** rồi chia cho số ảnh. Sau mỗi epoch, nếu validation loss thấp nhất từ trước tới giờ thì chép trạng thái model lại làm checkpoint.

In [ ]:
def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    total_loss, total_ok, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = F.cross_entropy(out, yb)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item() * len(yb); total_ok += (out.argmax(1) == yb).sum().item(); n += len(yb)
    return total_loss / n, total_ok / n

EPOCHS = 30
seed_all(0)
train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=1000)
model = make_model(); opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
hist, best_loss, t0 = [], float('inf'), time.time()
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, opt)
    va_loss, va_acc = run_epoch(model, valid_loader)
    hist.append((ep, tr_loss, va_loss, tr_acc, va_acc))
    if va_loss < best_loss:
        best_loss, best_ep, best_state = va_loss, ep, copy.deepcopy(model.state_dict())
H = pd.DataFrame(hist, columns=['epoch', 'train loss', 'val loss', 'train acc', 'val acc'])
show = H[H['epoch'].isin(sorted({1, 2, 3, 5, best_ep, 10, 15, 20, 25, EPOCHS}))]
table(show, f'Một số epoch ({EPOCHS} epoch, {time.time() - t0:.0f} giây)', fmt={c: '{:.4f}' for c in H.columns[1:]}, best='val loss', lower_is_better=True)
check('train loss giảm so với epoch đầu', H['train loss'].iloc[-1] < H['train loss'].iloc[0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(H['epoch'], H['train loss'], marker='o', ms=3, color=C['blue'], label='train')
axes[0].plot(H['epoch'], H['val loss'], marker='o', ms=3, color=C['clay'], label='validation')
axes[1].plot(H['epoch'], H['train acc'], marker='o', ms=3, color=C['blue'], label='train')
axes[1].plot(H['epoch'], H['val acc'], marker='o', ms=3, color=C['clay'], label='validation')
for ax in axes:
    ax.axvline(best_ep, color='#333', ls='--', lw=1)
axes[0].annotate(f'val loss thấp nhất\nepoch {best_ep}', (best_ep, best_loss), xytext=(14, 30), textcoords='offset points', fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='#333'))
axes[0].set(xlabel='epoch', ylabel='loss', title='Loss')
axes[1].set(xlabel='epoch', ylabel='accuracy', title='Accuracy')
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()
last = H.iloc[-1]
note(f"Validation loss thấp nhất ở epoch <b>{best_ep}</b> ({best_loss:.3f}). Đến epoch {EPOCHS}, train loss còn {last['train loss']:.3f} "
     f"nhưng validation loss lên {last['val loss']:.3f}: model đang học thuộc 10 000 ảnh train, khoảng cách hai đường loss là dấu hiệu overfitting.")

## 4. Chọn checkpoint theo validation

So hai model trên test: checkpoint ở epoch có validation loss thấp nhất, và model ở epoch cuối.

In [ ]:
test_loader = DataLoader(TensorDataset(X_te, y_te), batch_size=1000)
final_state = copy.deepcopy(model.state_dict())
rows = []
for name, state, ep in [(f'checkpoint tốt nhất (epoch {best_ep})', best_state, best_ep), (f'epoch cuối ({EPOCHS})', final_state, EPOCHS)]:
    model.load_state_dict(state)
    rows.append((name, *run_epoch(model, test_loader)))
ck = pd.DataFrame(rows, columns=['model', 'test loss', 'test acc'])
table(ck, f'Chấm trên {len(y_te)} ảnh test', fmt={'test loss': '{:.4f}', 'test acc': '{:.4f}'}, best='test loss', lower_is_better=True)
model.load_state_dict(best_state)
d_loss = ck['test loss'].iloc[1] / ck['test loss'].iloc[0] - 1
d_acc = ck['test acc'].iloc[0] - ck['test acc'].iloc[1]
msg = (f"Epoch cuối có test loss <b>{ck['test loss'].iloc[1]:.3f}</b> so với {ck['test loss'].iloc[0]:.3f} của checkpoint ({d_loss:+.0%}); "
       f"accuracy {ck['test acc'].iloc[1]:.3f} so với {ck['test acc'].iloc[0]:.3f}. ")
if d_loss > 0.1 and abs(d_acc) < 0.02:
    msg += "Loss tệ đi rõ còn accuracy gần như giữ nguyên: model overfit vẫn đoán đúng nhãn, nhưng khi sai thì sai rất tự tin."
note(msg)

## 5. Trung bình đúng theo mẫu

Chia 10 000 ảnh validation thành batch 3 000: ba batch đủ và một batch 1 000 ảnh. Lấy trung bình của **4 loss batch** thì batch cuối được tính nặng như batch 3 000 ảnh; đúng phải cộng theo từng ảnh rồi chia 10 000.

In [ ]:
model.eval()
sizes, means = [], []
with torch.no_grad():
    for xb, yb in DataLoader(TensorDataset(X_va, y_va), batch_size=3000):
        sizes.append(len(yb)); means.append(F.cross_entropy(model(xb.to(device)), yb.to(device)).item())
table(pd.DataFrame({'batch': range(1, len(sizes) + 1), 'số ảnh': sizes, 'loss trung bình của batch': means}), 'Bốn batch validation', fmt={'loss trung bình của batch': '{:.4f}'})
by_batch, by_sample = np.mean(means), np.average(means, weights=sizes)
table(pd.DataFrame([('trung bình 4 loss batch', by_batch), ('cộng theo từng ảnh rồi chia 10 000', by_sample)], columns=['cách tính', 'val loss']),
      'Hai cách lấy trung bình', fmt={'val loss': '{:.5f}'})
check('cách theo mẫu khớp run_epoch ở mục 3', np.isclose(by_sample, run_epoch(model, valid_loader)[0], atol=1e-5))
note(f'Hai cách lệch <b>{abs(by_batch - by_sample):.5f}</b>. Lệch nhỏ ở đây, nhưng sẽ lớn khi batch cuối rất nhỏ hoặc khác hẳn phần còn lại.')

## 6. Lưu và nạp lại

Checkpoint lưu `state_dict` cùng những thứ cần để dựng lại model và đọc kết quả: epoch, validation loss, cấu hình, tên lớp. Nạp vào một model mới tạo, ở chế độ `eval()`, phải ra đúng những dự đoán cũ.

In [ ]:
path = Path('best.pt')
torch.save({'model': best_state, 'epoch': best_ep, 'val_loss': best_loss, 'config': {'hidden': 512, 'lr': 1e-3}, 'classes': VI}, path)
ckpt = torch.load(path, map_location=device)
new_model = make_model(); new_model.load_state_dict(ckpt['model']); new_model.eval(); model.eval()
with torch.no_grad():
    old_pred = model(X_te.to(device)).argmax(1); new_pred = new_model(X_te.to(device)).argmax(1)
table(pd.DataFrame([('model', f"state_dict, {sum(v.numel() for v in ckpt['model'].values()):,} số"), ('epoch', ckpt['epoch']),
                    ('val_loss', f"{ckpt['val_loss']:.4f}"), ('config', str(ckpt['config'])), ('classes', f"{len(ckpt['classes'])} tên lớp"),
                    ('kích thước tệp', f'{path.stat().st_size / 2**20:.2f} MB')], columns=['khóa', 'nội dung']), f'Nội dung {path}')
check(f'model nạp lại cho đúng {len(new_pred)} dự đoán như model cũ', torch.equal(old_pred, new_pred))

## 7. Tái lập

Chạy cùng một đoạn huấn luyện ngắn ba lần trên CPU: hai lần seed 0, một lần seed 1. Seed quyết định trọng số khởi tạo và thứ tự xáo batch. Cùng seed, cùng code, cùng thiết bị thì loss phải trùng từng bước.

In [ ]:
def short_run(seed, steps=150):
    seed_all(seed)
    m = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
    o = torch.optim.SGD(m.parameters(), lr=0.1)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    out, it = [], iter(loader)
    for _ in range(steps):
        xb, yb = next(it)
        loss = F.cross_entropy(m(xb), yb); o.zero_grad(); loss.backward(); o.step(); out.append(loss.item())
    return np.array(out)

runs = {'seed 0, lần 1': short_run(0), 'seed 0, lần 2': short_run(0), 'seed 1': short_run(1)}
smooth = lambda v: pd.Series(v).rolling(10, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(9, 3.6))
for (name, v), c, ls, lw in zip(runs.items(), [C['blue'], C['sand'], C['clay']], ['-', '--', '-'], [3, 1.6, 1.6]):
    ax.plot(smooth(v), color=c, ls=ls, lw=lw, label=name)
ax.set(xlabel='bước', ylabel='loss (trung bình trượt 10 bước)', title='Hai lần seed 0 chồng khít lên nhau'); ax.legend()
plt.tight_layout(); plt.show()
diff_same = np.abs(runs['seed 0, lần 1'] - runs['seed 0, lần 2']).max()
diff_other = np.abs(runs['seed 0, lần 1'] - runs['seed 1']).max()
check(f'cùng seed: chênh lớn nhất {diff_same:.1e}', diff_same == 0)
note(f'Cùng seed chênh đúng <b>0</b>; đổi seed thì loss từng bước chênh tới {diff_other:.2f}, dù xu hướng chung vẫn giống.')

## 8. Model nhìn ảnh test

Checkpoint tốt nhất đoán 16 ảnh test ngẫu nhiên (xanh lá là đúng, đỏ cam là sai), rồi tới những ảnh nó sai mà **tự tin nhất**.

In [ ]:
model.load_state_dict(best_state); model.eval()
with torch.no_grad():
    prob = model(X_te.to(device)).softmax(1).cpu()
pred, conf = prob.argmax(1), prob.max(1).values

pick = torch.randperm(len(y_te), generator=torch.Generator().manual_seed(1))[:16]
fig, axes = plt.subplots(2, 8, figsize=(15, 4.6))
for ax, i in zip(axes.flat, pick):
    ok = pred[i] == y_te[i]
    ax.imshow(X_te[i].view(28, 28), cmap='gray_r'); ax.axis('off')
    ax.set_title(f'{VI[pred[i]]} ({conf[i]:.2f})', fontsize=9, color=C['green'] if ok else C['clay'])
fig.suptitle('Ảnh test ngẫu nhiên: lớp model đoán và xác suất', y=1.0)
plt.tight_layout(); plt.show()

wrong = torch.nonzero(pred != y_te).flatten()
top = wrong[conf[wrong].argsort(descending=True)[:8]]
fig, axes = plt.subplots(1, 8, figsize=(15, 2.6))
for ax, i in zip(axes, top):
    ax.imshow(X_te[i].view(28, 28), cmap='gray_r'); ax.axis('off')
    ax.set_title(f'thật: {VI[y_te[i]]}\nđoán: {VI[pred[i]]} ({conf[i]:.2f})', fontsize=9, color=C['clay'])
fig.suptitle('Sai mà tự tin nhất', y=1.08)
plt.tight_layout(); plt.show()
pair = pd.Series([(VI[y_te[i]], VI[pred[i]]) for i in wrong.tolist()]).value_counts()
note(f'{len(wrong)} ảnh test bị đoán sai. Cặp nhầm nhiều nhất: <b>{pair.index[0][0]}</b> bị đoán thành <b>{pair.index[0][1]}</b> ({pair.iloc[0]} ảnh).')

## 9. Tóm tắt

In [ ]:
summary = [('loss bước đầu (gần ln 10 = 2.303)', f'{one_batch[0]:.3f}'),
           ('epoch có val loss thấp nhất', f'{best_ep} / {EPOCHS}'),
           ('test acc: checkpoint tốt nhất / epoch cuối', f"{ck['test acc'].iloc[0]:.4f} / {ck['test acc'].iloc[1]:.4f}"),
           ('test loss: checkpoint tốt nhất / epoch cuối', f"{ck['test loss'].iloc[0]:.4f} / {ck['test loss'].iloc[1]:.4f}"),
           ('nạp lại checkpoint cho cùng dự đoán', 'có' if torch.equal(old_pred, new_pred) else 'không'),
           ('cùng seed, chênh loss lớn nhất', f'{diff_same:.1e}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** cho model học cả 50 000 ảnh thay vì 10 000 rồi xem khoảng cách hai đường loss; thêm `nn.Dropout(0.3)` sau mỗi `ReLU`; đổi `lr=1e-3` thành `1e-2` ở mục 3 để thấy loss dao động.